# RAG-System

RAG (Retrieval Augmented Generation) bedeutet, dass Benutzeranfragen mit passenden Context-Informationen ergänzt werden, so dass ein generatives Sprachmodell sehr spezifische Antworten generieren und sogar die Quellen angeben kann. Als Informations-Quellen kommen z.B. eigene Dokumente in Frage, die nach entsprechender Aufbereitung in einer Vektor-Datenbank gespeichert werden. Auf diese  Weise läßt sich das aufwändige Nachtrainieren des Sprachmodells vermeiden.

In diesem Projekt werden folgende Software-Pakete verwendet:
* Betriebssystem "Windows" (auch andere möglich)
* Programmiersprache:       Python (3.9.4) (https://www.python.org/downloads)
* Python-Modul:             gpt4all (2.8.2)
* Embeddingmodell:          all-MiniLM-L6-v2.gguf2.f16.gguf
* generatives Sprachmodell: Meta-Llama-3-8B-Instruct.Q4_0.gguf
* Vektordatenbank:          qdrant (v1.15.3)  (https://github.com/qdrant/qdrant/releases/tag/v1.15.3)
* Qdrant-Weboberfläche:     qdrant-client (v0.2.3)  /https://github.com/qdrant/qdrant-web-ui/releases)

Installations-Anleitungen für obige Pakete folgen, vorausgesetzt werden lediglich:
* Python (mindestens 3.9.4)
* pip (mindestens 20.2.3)
* jupyter notebook (mindestens 2.17.0)

# 1. Vorbereitungen

## 1.1 Python-Pakete installieren

In [ ]:
# Bitte einmal ausführen, um die Python-Pakete zu installieren
!pip install gpt4all
!pip install qdrant-client

## 1.2 Datenbank installieren
1. Herunterladen:
    - qdrant-Datenbank (_qdrant-x86_64-pc-windows-msvc.zip_): https://github.com/qdrant/qdrant/releases/tag/v1.15.3
    - qdrant-Weboberfläche (_dist-qdrant.zip_): https://github.com/qdrant/qdrant-web-ui/releases (v0.2.3)
4. neuen leeren Ordner _Qdrant_ anlegen
5. _qdrant-x86_64-pc-windows-msvc.zip_ nach Ordner _Qdrant_ auspacken!
3. Web-Oberfläche hinzufügen:
   - _dist-qdrant.zip_ nach _Qdrant_ auspacken
   - _dist_ in _static_ umbenennen
4. Datei _qdrant.exe_ starten
5. Web-UI öffnen über URL: http://localhost:6333/dashboard

## 1.3 Collection anlegen

### 1.3.1 Mit Web-UI
1. Menü _Collections_
2. Button _Create Collection_
3. Collection name: _Flugunfälle_
4. Use case: _Global Search_
5. What to use for search: _Simple Single embedding_
6. Choose Dimensions: 384
7. Choose metric: Dot
8. Button _Finish_

### 1.3.2 Mit Python-Skript

In [ ]:
# Collection anlegen
from qdrant_client.models import Distance, VectorParams
from qdrant_client import QdrantClient

client = QdrantClient(url="http://localhost:6333")
client.create_collection(
    collection_name="Flugunfälle",
    vectors_config=VectorParams(size=384, distance=Distance.DOT),
)

# 2. Dokument in Wissensdatenbank speichern
In diesem Abschnitt soll ein Text-Dokument In Chunks zerlegt und zusammen mit dem Embedding in der Vektordatenbank gespeichert werden.

Die Aufgabe zerlegt sich in folgende Teilschritte:
* Text absatzweise in Chunks zerlegen (Funktion: get_chunks)
* Embeddings zu den Chunks berechnen (Funktion: get_embeddings)
* Chunks und Embeddings für Vektordatenbank vorbereiten (Funktion: get_point_structs)
* Daten in Vektordatenbank speichern (Funktion: save_point_structs)

## 2.1 Funktion: _get_chunks(file_name)_
__Auftrag:__ Schreibe die Funktion _get_chunks(file_name)_ !

Sie soll:
- die Text-Datei öffnen und einlesen 
- den Datei-Inhalt in Chunks (Absätze) zerlegen
- eine Chunk-Liste (Teil-Strings) zurückgeben

Wir gehen davon aus, dass der Text als UTF-8-Datei vorliegt und die Absätze durch eine Leerzeile ('\n\n') voneinander getrennt sind. Falls nicht, dann muss der Text vorher manuell mit einem Text-Editor (z.B. Notepad++) aufbereitet werden.

__Programmier-Hinweise:__
| Befehl | Beschreibung |
|--|--|
| fin = open(file_name, 'r', encoding='utf-8') | Datei zum Lesen öffnen                 |
| fin.close()                                  | Datei schließen                        |
| text = fin.read()                            | ganze Datei in Stringvariable einlesen |
| parts = t.split(';')                         | String _t_ anhand ';' in mehrere Teil-Strings aufteilen |

In [ ]:
def get_chunks(file_name):
    # hier vervollständigen



__Teste die Funktion!__

In [ ]:
chunks = get_chunks('Gleitschirm-Unfallbericht-2020.txt')
print(*chunks, sep='\n\n')

## 2.2 Funktion: _get_embeddings(chunks, embedder)_

__Auftrag:__ Schreibe die Funktion _get_embeddings(chunks)_ !

Sie soll:
* zu jedem Chunk das Embedding berechnen
* alle Embeddings in einer Liste sammeln (gleiche Reihenfolge!)
* Embeddings-Liste zurückgeben

__Programmier-Hinweise:__
| Befehl | Beschreibung |
|--|--|
| list = [] | leere Liste anlegen |
| list.append(data) | Eintrag an Liste anhängen |
| embedder = gpt4all.Embed4All() | Embedding-Modell bereitstellen |
| embedding = embedder.embed(chunk) | Embedding für einen Chunk berechnen |
| for element in list: | Elemente in Liste _list_ durchiterieren (foreach-Schleife) |

In [ ]:
def get_embeddings(chunks, embedder):
    # hier vervollständigen



__Teste die Funktion!__

In [ ]:
import gpt4all
embedder = gpt4all.Embed4All()

embeddings = get_embeddings(chunks, embedder)
print(embeddings)

## 2.3 Funktion: _get_point_structs(chunks, embeddings)_

__Auftrag:__ Schreibe die Funktion _get_point_structs(chunks, embeddings)_ !

Sie soll:
* aus den Listen _chunks_ und _embeddings_ eine Liste von PointStruct-Objekten erstellen
* die PointStruct-Liste zurückgeben 

__Programmier-Hinweise:__

ein PointStrukt-Objekt mit erstellen:

`point_struct = PointStruct(id=7, vector=embedding, payload={"chunk": chunk})`

Jedes PointStruct-Objekt benötigt eine eigene id!

In [ ]:
def get_point_structs(chunks, embeddings):
    # hier vervollständigen



__Teste die Funktion!__

In [ ]:
from qdrant_client.models import PointStruct

point_structs = get_point_structs(chunks, embeddings)
print(*point_structs, sep='\n\n')

## 2.4 Funktion: _save_point_structs(db_url, db_collection, point_structs)_

__Auftrag:__ Schreibe die Funktion _save_to_database(db_url, db_collection, point_structs)_ !

Sie soll:
* eine Verbindung zum Datenbank-Server herstellen
* die PointStruct-Objekte in der Datenbank speichern
* das operation_info-Objekt zurückgeben

__Programmier-Hinweise:__

Datenbank-Client-Verbindung zu "localhost" auf Port 6333 herstellen:
<pre>
    client = QdrantClient(db_url)
</pre>

PointStruct-Liste p an Datenbank-Server übermitteln und in Collection "Flugunfälle" speichern. Der Erfolg kann dem operation_info-Objekt entnommen werden.
<pre>
    operation_info = client.upsert(
    collection_name="Flugunfälle",
    wait=True,
    points=p
)
</pre>

In [ ]:
def save_point_structs(db_url, db_collection, point_structs):
    # hier vervollständigen



__Teste die Funktion!__

In [ ]:
from qdrant_client import QdrantClient

db_url = 'http://localhost:6333'
db_collection = 'Flugunfälle'

operation_info = save_point_structs(db_url, db_collection, point_structs)
print(operation_info)

__Überprüfe über das WebUI, ob die Daten in der Datenbank gespeichert sind!__

## 2.5 Skript: make_embeddings.py
__Auftrag:__ Fasse die obigen Funktionen so zu einem Python-Skript zusammen, dass ein Text-Dokument hinterher in der Vektordatenbank gespeichert ist!

In [ ]:
# make_embeddings.py
# hier vervollständigen



# 3. ChatBot

Hier soll ein ChatBot programmiert werden, der eine Frage vom Benutzer entgegennimmt, sie in ein Embedding umrechnet und in der Vektor-Datenbank nach den zwei am besten passenden Chunks sucht. Diese Chunks werden dann zusammen mit einem System-Prompt und der Benutzer-Frage zu einem Gesamt-Prompt zusammengesetzt.

Der Gesamt-Prompt wird an ein generatives Sprachmodell (z.B.: Meta-Llama-3-8B-Instruct.Q4_0.gguf) übergeben und der generierte Text dem Benutzer angezeigt. Die Antwort sollte spezielle Informationen aus dem ursprünglichen Text-File enthalten.

Die Aufgabe zerlegt sich in folgende Teilschritte:
* Kontext-Informationen für Benutzerfrage mit Hilfe der Vektordatenbank erstellen (Funktion: get_context)
* Frage und Kontext zu einem Prompt zusammensetzen (Funktion: create_prompt)
* Prompt an generatives Sprachmodell übergeben und Antwort abholen (Funktion: generate_answer)

## 3.1 Funktion: _get_context(db_url, db_collection, user_question, embedder)_

__Auftrag:__ Schreibe die Funktion _get_context(user_question)_ !

Sie soll:
* das Embedding zur Benutzer-Frage berechnen
* die Datenbank nach den zwei am besten passenden Chunks durchsuchen
* die beiden Chunks in einem einzigen String zurückgeben

__Programmier-Hinweise:__

Vektor-Datenbank _client_ nach passenden Chunks durchsuchen und höchstens 5 zurückgeben:
<pre>
search_result = client.query_points(
    collection_name="MeineKollektion",
    query=question_embedding,
    with_payload=True,
    limit=5
)
</pre>

3ten Chunk aus Suchergebnissen auslesen:
<pre>
search_result.points[2].payload['chunk']
</pre>

In [ ]:
def get_context(db_url, db_collection, user_question, embedder):
    # hier vervollständigen



__Teste die Funktion!__

In [ ]:
import gpt4all
from qdrant_client import QdrantClient

db_url = 'http://localhost:6333'
db_collection = 'Flugunfälle'
question = 'Wie viele Unfälle gab es beim Start?'
embedder = gpt4all.Embed4All()

context = get_context(db_url, db_collection, question, embedder)
print(context)

## 3.2 Funktion: _create_prompt(context, question)_

__Auftrag:__ Schreibe die Funktion _get_prompt(context, question)_ !

Sie soll:
* die Kontext-Information und die Benutzerfrage zu einem Gesamtprompt zusammenfügen.
* den Gesamt-Prompt zurückgeben

__Programmier-Hinweise:__

Schema für einen Gesamtprompt für lLama-Modelle:
<pre>
<|begin_of_text|>
<|start_header_id|>system<|end_header_id|>
Du bist ein hilfreicher Assistent, der Fragen basierend auf dem Kontext beantwortet.
Antworte kurz und knapp.
<|eot_id|>

<|start_header_id|>user<|end_header_id|>
Kontext:
{context}

Frage:
{question}
<|eot_id|>

<|start_header_id|>assistant<|end_header_id|>
</pre>

In [ ]:
def create_prompt(context, question):
    # hier vervollständigen



__Teste die Funktion!__

In [ ]:
prompt = create_prompt(context, question)
print(prompt)

## 3.3 Funktion: _generate_answer(prompt, model)_

In [ ]:
def generate_answer(prompt, model):
    gen = model.generate(prompt, streaming=True, temp=0.0)
    answer = ""
    for token in gen:
        print(token, end="", flush=True)
        answer += token
        if answer.find('<|eot_id|>') >= 0:
            gen.close()

__Teste die Funktion!__

In [ ]:
import gpt4all

model = gpt4all.GPT4All('Meta-Llama-3-8B-Instruct.Q4_0.gguf', allow_download=True)
generate_answer(prompt, model)

## 3.4 Skript: _rag_chatbot.py_
__Auftrag:__ Fasse die obigen Funktionen so zu einem Python-Skript zusammen, dass der Benutzer eine Frage eingeben kann und darauf eine passende Antwort erhält!

In [ ]:
# rag_chatbot.py
# hier vervollständigen



Beispiele für Fragen:
- Wie viele Unfälle gab es beim Start?
- Was sind die Hauptursachen für Strömungsabrisse?
- Wieviele Unfälle gab es bei der Landeinteilung?
- Was sind die häufigsten Unfallursachen beim Start?
- Gab es Kollisionen?